In [1]:
!pip install dagshub
!pip install mlflow 
!pip install dagshub mlflow imbalanced-learn --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 6.0 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: dacite
    Found existing installation: dacite 1.9.2
    Uninstalling dacite-1.9.2:
      Successfully uninstalled dacite-1.9.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import os
import dagshub
import mlflow
import dagshub.auth

# Set token as environment variable (replace with actual token)
os.environ['DAGSHUB_TOKEN'] = '237c5c1b853b5aee083fa279da4224f289a29cbb'

# Initialize DagsHub
dagshub.auth.add_app_token(token=os.environ['DAGSHUB_TOKEN'])
dagshub.init(repo_owner='slomi23', repo_name='slomi23ML2', mlflow=True)



/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


Accessing as slomi23

Initialized MLflow to track repo "slomi23/slomi23ML2"

Repository slomi23/slomi23ML2 initialized!

In [3]:
import pandas as pd
import numpy as np
train_identity=pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
train_transaction=pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")

test_identity=pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv")
test_transaction=pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")

In [4]:
trainset=pd.merge(train_identity, train_transaction, on="TransactionID", how="left")
trainset.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
from sklearn.model_selection import train_test_split

# Assuming your target column is 'isFraud'
X = trainset.drop('isFraud', axis=1)
y = trainset['isFraud']

# Split the data - 80% train, 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X, 
    y, 
    test_size=0.2,  # 20% for validation
    random_state=42,  # For reproducibility
    stratify=y  # Important for classification datasets with imbalanced classes
)

print(f"Training set size: {len(X_train)} samples")
print(f"Validation set size: {len(X_val)} samples")
print(f"Training set fraud percentage: {y_train.mean():.2%}")
print(f"Validation set fraud percentage: {y_val.mean():.2%}")

Training set size: 115386 samples
Validation set size: 28847 samples
Training set fraud percentage: 7.85%
Validation set fraud percentage: 7.85%


In [8]:
from sklearn.base import BaseEstimator, TransformerMixin
import shap
class TargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, categorical_columns, smoothing=1.0):
        self.categorical_columns = categorical_columns
        self.smoothing = smoothing
        self.target_means = {}
        self.global_mean = None
        
    def fit(self, X, y):
        self.global_mean = y.mean()
        
        for col in self.categorical_columns:
            # Calculate target mean for each category
            cat_counts = X[col].value_counts()
            cat_target_means = y.groupby(X[col]).mean()
            
            # Apply smoothing
            self.target_means[col] = (
                (cat_target_means * cat_counts + self.global_mean * self.smoothing) / 
                (cat_counts + self.smoothing)
            )
        return self
    
    def transform(self, X):
        X_encoded = X.copy()
        
        for col in self.categorical_columns:
            # Map categories to target-encoded values
            X_encoded[col] = X[col].map(self.target_means[col]).fillna(self.global_mean)
        
        return X_encoded

In [15]:
#i couldnt manage to loaad and run the model

Current tracking URI: file:./mlruns

Available experiments:
Experiment ID: 0, Name: Default

Runs in experiment 'Default':


MlflowException: Run '684a550ff8824611a0f63fdde9362133' not found